In [3]:
import requests
import json
import pandas as pd
import csv
import re

# Membaca file madaniyah.csv untuk mendapatkan nomor surah
madaniyah_chapters = []
with open('../madaniyah.csv', 'r') as file:
    csv_reader = csv.reader(file)
    next(csv_reader)  # Melewati header
    for row in csv_reader:
        madaniyah_chapters.append(int(row[1]))

# Dictionary untuk menyimpan semua DataFrame
dfs = {}

# Melakukan request untuk setiap chapter di madaniyah.csv
for chapter in madaniyah_chapters:
    # URL untuk chapter yang sedang diproses
    url = f"https://api.qurancdn.com/api/v4/verses/by_chapter/{chapter}?language=id&translations=33,134&words=0&fields=text_uthmani&per_page=300"
    
    # Melakukan request ke API
    response = requests.get(url)
    
    # Memeriksa apakah request berhasil
    if response.status_code == 200:
        # Parse JSON response
        data = response.json()
        
        # Membuat list untuk menyimpan data
        arab_texts = []
        king_fahd_texts = []
        kemenag_texts = []
        verse_number = []
        
        # Mengekstrak data yang dibutuhkan
        for verse in data['verses']:
            # Mengambil teks Arab
            arab_texts.append(verse['text_uthmani'])
            
            # Mengambil verse key (nomor ayat)
            verse_number.append(verse['verse_number'])
            
            # Mengambil terjemahan
            for translation in verse['translations']:
                if translation['resource_id'] == 134:
                    # Membersihkan superscript footnote dari teks King Fahd
                    clean_text = re.sub(r'<sup foot_note=\d+>\d+</sup>', '', translation['text'])
                    king_fahd_texts.append(clean_text)
                elif translation['resource_id'] == 33:
                    # Membersihkan superscript footnote dari teks Kemenag
                    clean_text = re.sub(r'<sup foot_note=\d+>\d+</sup>', '', translation['text'])
                    kemenag_texts.append(clean_text)
        
        # Membuat DataFrame
        df_arab = pd.DataFrame({
            'verse_number': verse_number,
            'arab': arab_texts
        })

        df_kingfahd = pd.DataFrame({
            'verse_number': verse_number,
            'king_fahd': king_fahd_texts
        })

        df_kemenag = pd.DataFrame({
            'verse_number': verse_number,
            'kemenag': kemenag_texts
        })

        # Menampilkan informasi sebelum menyimpan
        print(f"Chapter {chapter}: {len(verse_number)} verses processed")
        
        # Menyimpan DataFrame ke file CSV
        df_arab.to_excel(f'../data/arab/chapter_{chapter}.xlsx', index=False)
        df_kingfahd.to_csv(f'../data/king_fahd/chapter_{chapter}.csv', index=False)
        df_kemenag.to_csv(f'../data/kemenag/chapter_{chapter}.csv', index=False)
        
        print(f"Chapter {chapter} saved successfully")


Chapter 2: 286 verses processed
Chapter 2 saved successfully
Chapter 3: 200 verses processed
Chapter 3 saved successfully
Chapter 4: 176 verses processed
Chapter 4 saved successfully
Chapter 5: 120 verses processed
Chapter 5 saved successfully
Chapter 9: 129 verses processed
Chapter 9 saved successfully
Chapter 24: 64 verses processed
Chapter 24 saved successfully
Chapter 33: 73 verses processed
Chapter 33 saved successfully
Chapter 47: 38 verses processed
Chapter 47 saved successfully
Chapter 48: 29 verses processed
Chapter 48 saved successfully


SSLError: HTTPSConnectionPool(host='api.qurancdn.com', port=443): Max retries exceeded with url: /api/v4/verses/by_chapter/49?language=id&translations=33,134&words=0&fields=text_uthmani&per_page=300 (Caused by SSLError(SSLEOFError(8, '[SSL: UNEXPECTED_EOF_WHILE_READING] EOF occurred in violation of protocol (_ssl.c:1007)')))

In [4]:
df_arab = pd.read_excel('../data/arab/chapter_2.xlsx')
df_kemenag = pd.read_csv('../data/kemenag/chapter_2.csv')
df_kingfahd = pd.read_csv('../data/king_fahd/chapter_2.csv')

df_arab.head()

,verse_number,arab
0,1,الٓمٓ
1,2,ذَٰلِكَ ٱلْكِتَـٰبُ لَا رَيْبَ ۛ فِيهِ ۛ هُدًى...
2,3,ٱلَّذِينَ يُؤْمِنُونَ بِٱلْغَيْبِ وَيُقِيمُونَ...
3,4,وَٱلَّذِينَ يُؤْمِنُونَ بِمَآ أُنزِلَ إِلَيْك...
4,5,أُو۟لَـٰٓئِكَ عَلَىٰ هُدًى مِّن رَّبِّهِمْ ۖ و...


In [5]:
df_kemenag.head()

,verse_number,kemenag
0,1,Alif Lām Mīm.
1,2,Kitab (Alquran) ini tidak ada keraguan padanya...
2,3,"(Yaitu) mereka yang beriman kepada yang gaib, ..."
3,4,dan mereka beriman kepada (Alquran) yang ditur...
4,5,Merekalah yang mendapat petunjuk dari Tuhannya...


In [6]:
df_kingfahd.head()

,verse_number,king_fahd
0,1,Alif Lām Mīm.
1,2,Kitab (Al-Qur`ān) ini tidak ada keraguan padan...
2,3,"(yaitu) mereka yang beriman kepada yang gaib, ..."
3,4,dan mereka yang beriman kepada Kitab (Al-Qur`ā...
4,5,Mereka itulah yang tetap mendapat petunjuk dar...


In [ ]:
df_translated = {}

for chapter in madaniyah_chapters:
    df = pd.read_excel(f'../data/arab_translated/chapter_{chapter}.xlsx')
    df_translated[chapter] = df

In [10]:
df_translated[2]

,nomor_ayat,Arab
0,1,nyeri
1,2,Kitab ini tidak ada keraguan padanya; petunju...
2,3,"Orang-orang yang beriman kepada yang gaib, me..."
3,4,Dan orang-orang yang beriman kepada kitab yan...
4,5,Mereka itulah yang mendapat petunjuk dari Tuh...
...,...,...
281,282,"Hai orang-orang yang beriman, jika kamu beruta..."
282,283,۞ Dan jika kamu dalam perjalanan dan tidak men...
283,284,Kepunyaan Allah-lah apa yang ada di langit da...
284,285,Rasulullah telah beriman kepada apa yang ditur...


In [12]:
print(df_translated[2].columns)

Index(['nomor_ayat', ' Arab'], dtype='object')


In [13]:
for chapter in df_translated:
    df_translated[chapter] = df_translated[chapter].rename(columns={' Arab': 'google_translate'})

df_translated[2]

,nomor_ayat,google_translate
0,1,nyeri
1,2,Kitab ini tidak ada keraguan padanya; petunju...
2,3,"Orang-orang yang beriman kepada yang gaib, me..."
3,4,Dan orang-orang yang beriman kepada kitab yan...
4,5,Mereka itulah yang mendapat petunjuk dari Tuh...
...,...,...
281,282,"Hai orang-orang yang beriman, jika kamu beruta..."
282,283,۞ Dan jika kamu dalam perjalanan dan tidak men...
283,284,Kepunyaan Allah-lah apa yang ada di langit da...
284,285,Rasulullah telah beriman kepada apa yang ditur...


In [14]:
for chapter, df in df_translated.items():
    df.to_csv(f'../data/google_translate/chapter_{chapter}.csv', index=False)

In [15]:
df_google_translate = pd.read_csv('../data/google_translate/chapter_2.csv')
df_google_translate.head()

,nomor_ayat,google_translate
0,1,nyeri
1,2,Kitab ini tidak ada keraguan padanya; petunju...
2,3,"Orang-orang yang beriman kepada yang gaib, me..."
3,4,Dan orang-orang yang beriman kepada kitab yan...
4,5,Mereka itulah yang mendapat petunjuk dari Tuh...
